# Copenhagen OSM vs official accessibility

This notebook is the runnable version of the project pipeline. We kept the real code in `src/` because it is easier to test, reuse, and rerun from the command line. The notebook gives the same workflow in smaller chunks, with notes about what each step is doing.

Run the cells in order for a fresh build. If the processed files already exist, you can jump to the later diagnostic and mapping sections.

## setup

The notebook can be opened from the project root or from the `notebooks/` folder. This cell moves to the project root and defines one small helper for running a stage.

In [1]:
from pathlib import Path
import os
import runpy
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

SRC_DIR = PROJECT_ROOT / "src"
src_path = str(SRC_DIR)
if src_path not in sys.path:
    sys.path.insert(0, src_path)


def run_stage(script_name: str, *args: str) -> None:
    script_path = SRC_DIR / script_name
    old_argv = sys.argv[:]
    sys.argv = [str(script_path), *args]
    try:
        print(f"Running {script_name}")
        runpy.run_path(str(script_path), run_name="__main__")
        print(f"Finished {script_name}")
    finally:
        sys.argv = old_argv


## quick file check

The OSM PBF is deliberately local. This check fails early if the file is missing, which is better than discovering the problem halfway through the run.

In [2]:
from pathlib import Path

pbf_path = PROJECT_ROOT / "data" / "raw" / "osm" / "denmark-latest.osm.pbf"
if not pbf_path.exists():
    raise FileNotFoundError(
        f"Place the Denmark OSM PBF at {pbf_path}. The project does not download it."
    )
print(f"Found local PBF: {pbf_path}")

Found local PBF: c:\Users\dubst\Desktop\DataScience\Geospatial\Project\data\raw\osm\denmark-latest.osm.pbf


## Download the city data

This pulls the four Copenhagen datasets from Open Data DK and keeps the CKAN metadata. We like saving the source links here because open-data portals change more often than anyone wants.

In [3]:
run_stage("01_download_official_data.py")

Running 01_download_official_data.py
Saved metadata for 4 official datasets.
Finished 01_download_official_data.py


## Clean the official layers

The official files come in with their own field names and geometry quirks. This step inspects them, clips them to Copenhagen, and writes a common amenity schema.

In [4]:
run_stage("03_clean_official_amenities.py")

Running 03_clean_official_amenities.py
Bydele columns: ['id', 'bydel_nr', 'navn', 'areal_m2', 'ogc_fid', 'geometry']
libraries columns: ['navn', 'adresse', 'post_nr', 'postdistrikt', 'e_mail', 'link', 'id', 'kkorgnr', 'geometry']
playgrounds columns: ['id', 'legeplads_id', 'navn', 'adressebeskrivelse', 'bydel', 'legeplads_type', 'aldersgruppe', 'ejer', 'beskrivelse', 'link', 'graffitirenhold', 'graffitirenhold_id', 'lokaludvalgnr', 'lokaludvalgnavn', 'geometry']
sports_facilities columns: ['navn', 'adresse', 'post_nr', 'postdistrikt', 'e_mail', 'link', 'id', 'geometry']
Saved official inspection table with 4 rows.
Finished 03_clean_official_amenities.py


## Extract OSM from the local PBF

This uses the local Denmark PBF only. No Overpass, no Geofabrik download. The same OSM walking network will be used for both accessibility scenarios.

In [5]:
run_stage("02_extract_osm_data.py")

Running 02_extract_osm_data.py
Extracting OSM libraries: {'amenity': ['library']}
Saved 31 OSM libraries features.
Extracting OSM playgrounds: {'leisure': ['playground']}
Saved 617 OSM playgrounds features.
Extracting OSM sports_facilities: {'leisure': ['sports_centre', 'sports_hall']}
Saved 112 OSM sports_facilities features.
Extracting OSM walking network from local PBF.
Saved walking network: 120615 nodes, 138693 edges.
Finished 02_extract_osm_data.py


## Clean the OSM amenities

OSM points and polygons get converted into the same schema as the official amenities. For routing, polygons become representative points, while the original geometries stay on disk.

In [6]:
run_stage("04_clean_osm_amenities.py")

Running 04_clean_osm_amenities.py
OSM libraries columns: ['visible', 'timestamp', 'changeset', 'tags', 'lat', 'lon', 'id', 'version', 'email', 'name', 'opening_hours', 'operator', 'phone', 'website', 'amenity', 'internet_access', 'wikipedia', 'osm_type', 'building', 'building:levels', 'source', 'start_date', 'geometry']
OSM playgrounds columns: ['visible', 'timestamp', 'changeset', 'tags', 'lat', 'lon', 'id', 'version', 'name', 'opening_hours', 'website', 'leisure', 'playground', 'osm_type', 'operator', 'phone', 'geometry']
OSM sports_facilities columns: ['visible', 'timestamp', 'changeset', 'tags', 'lat', 'lon', 'id', 'version', 'email', 'name', 'opening_hours', 'operator', 'phone', 'url', 'website', 'leisure', 'outdoor_seating', 'osm_type', 'ref', 'geometry']
Saved cleaned OSM amenity layers for 3 amenity types.
Finished 04_clean_osm_amenities.py


## Prepare the walking network

The walking edges are projected to EPSG:25832, lengths are measured in metres, and walking time is calculated at 5 km/h.

In [7]:
run_stage("05_prepare_walking_network.py")

Running 05_prepare_walking_network.py
Saved GraphML: C:\Users\dubst\Desktop\DataScience\Geospatial\Project\data\processed\osm_walking_graph.graphml
Prepared walking network: 113861 nodes, 130733 undirected edges.
Finished 05_prepare_walking_network.py


## Build the 500 m origin grid

The grid is intentionally simple. It gives us a regular set of origins, but the edge cells need care later because Copenhagen has water, islands, and awkward boundaries.

In [8]:
run_stage("06_create_origin_grid.py")

Running 06_create_origin_grid.py
Saved 475 origin grid cells and centroid points.
Finished 06_create_origin_grid.py


## Match OSM and official POIs

Before routing, this checks how many OSM amenities match an official amenity nearby. It is a useful sanity check on completeness and classification.

In [9]:
run_stage("07_match_osm_to_official.py")

Running 07_match_osm_to_official.py
Saved C:\Users\dubst\Desktop\DataScience\Geospatial\Project\outputs\maps\matched_unmatched_pois.html
Saved POI completeness summary and match details.
Finished 07_match_osm_to_official.py


## Compute baseline accessibility

For each source and amenity type, this uses multi-source Dijkstra to get the walking time from every origin to the nearest amenity.

In [10]:
run_stage("08_compute_accessibility.py")

Running 08_compute_accessibility.py
Saved accessibility comparison for 1425 origin/category rows.
Finished 08_compute_accessibility.py


## Classify the baseline differences

This labels each origin as agreement, OSM false access, or OSM hidden access. Those labels make the maps easier to read than raw time differences alone.

In [11]:
run_stage("09_compare_accessibility.py")

Running 09_compare_accessibility.py
Saved classified accessibility, composite disagreement, district summary, and limitations notes.
Finished 09_compare_accessibility.py


## Make the baseline maps

These are the first-pass maps. They are still useful for comparison, even though the origin-quality checks below produce the recommended final maps.

In [12]:
run_stage("10_make_maps.py")

Running 10_make_maps.py


c:\Users\dubst\Desktop\DataScience\Geospatial\Project\src\10_make_maps.py:77: UserWarning: Legend does not support handles for PatchCollection instances.
See: https://matplotlib.org/stable/tutorials/intermediate/legend_guide.html#implementing-a-custom-legend-handler
  ax.legend(loc="lower left", frameon=True)
c:\Users\dubst\Desktop\DataScience\Geospatial\Project\src\10_make_maps.py:77: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc="lower left", frameon=True)
c:\Users\dubst\Desktop\DataScience\Geospatial\Project\src\10_make_maps.py:77: UserWarning: Legend does not support handles for PatchCollection instances.
See: https://matplotlib.org/stable/tutorials/intermediate/legend_guide.html#implementing-a-custom-legend-handler
  ax.legend(loc="lower left", frameon=True)
c:\Users\dubst\Desktop\DataScience\Geospatial\Project\src\10_make_maps.py:77: UserWarn

Saved static PNG maps in C:\Users\dubst\Desktop\DataScience\Geospatial\Project\outputs\figures.
Finished 10_make_maps.py


## Diagnose origin quality

The baseline run left too many origins unassigned to districts. This script reconstructs that problem and checks which origins sit on boundary or harbour-like edge cells.

In [13]:
run_stage("11_diagnose_origin_quality.py")

Running 11_diagnose_origin_quality.py
Saved origin-quality diagnostics: 103 reconstructed previously unassigned origins.
Finished 11_diagnose_origin_quality.py


## Assign districts by overlap

Centroid assignment is brittle for coastal grid cells. This assigns each grid cell to the Bydele polygon with the largest area overlap.

In [14]:
run_stage("12_assign_districts_by_overlap.py")

Running 12_assign_districts_by_overlap.py
Saved largest-overlap district assignment for 475 origin grid cells.
Finished 12_assign_districts_by_overlap.py


## Check snapping distances

Snapping hundreds of metres away from a grid centroid can bend the accessibility result. This step flags those origins and makes a histogram plus an outlier map.

In [15]:
run_stage("13_diagnose_snapping_outliers.py")

Running 13_diagnose_snapping_outliers.py
Saved snapping diagnostics: 150 origins >50 m, 87 origins >100 m.
Finished 13_diagnose_snapping_outliers.py


## Create full and cleaned origin sets

The baseline stays intact. The clean100 and clean250 sets remove origins with large snapping distances and very weak district overlap.

In [16]:
run_stage("14_create_clean_origin_set.py")

Running 14_create_clean_origin_set.py
Saved full, clean100, and clean250 origin sets.
Finished 14_create_clean_origin_set.py


## Rerun accessibility on the cleaned origins

The walking network and amenity layers stay fixed here. Only the origins change, which keeps the sensitivity test clean.

In [17]:
run_stage("15_rerun_accessibility_for_clean_origins.py")

Running 15_rerun_accessibility_for_clean_origins.py
Saved accessibility for full: 475 origins.
Saved accessibility for clean100: 387 origins.
Saved accessibility for clean250: 428 origins.
Finished 15_rerun_accessibility_for_clean_origins.py


## Rebuild district summaries

District summaries now use the largest-overlap assignment. Any origin with no real district overlap gets reported separately instead of being hidden in an Unassigned bucket.

In [18]:
run_stage("16_rerun_district_summaries.py")

Running 16_rerun_district_summaries.py
Saved corrected district summary for full: 10 districts.
Saved corrected district summary for clean100: 10 districts.
Saved corrected district summary for clean250: 10 districts.
Finished 16_rerun_district_summaries.py


## Compare baseline and cleaned results

This is the robustness check. If the same story holds after cleaning, the findings are much easier to defend.

In [19]:
run_stage("17_compare_baseline_vs_cleaned.py")

Running 17_compare_baseline_vs_cleaned.py
Saved robustness summaries, comparison table, barplot, and interpretation notes.
Finished 17_compare_baseline_vs_cleaned.py


## Make the cleaned maps

The clean100 maps are the ones We would use in the write-up, with the baseline maps kept as a check.

In [20]:
run_stage("18_make_cleaned_maps.py")

Running 18_make_cleaned_maps.py
Saved clean100 maps.
Finished 18_make_cleaned_maps.py


In [21]:
import pandas as pd

tables = [
    "outputs/tables/origin_quality_diagnostic_counts.csv",
    "outputs/tables/origin_cleaning_summary.csv",
    "outputs/tables/robustness_summary_by_origin_set.csv",
]

for table in tables:
    print(f"\n{table}")
    display(pd.read_csv(PROJECT_ROOT / table))


outputs/tables/origin_quality_diagnostic_counts.csv


,metric,value
0,origin_count,475
1,origins_in_accessibility_table,475
2,reconstructed_previous_unassigned_origins,103
3,district_summary_unassigned_n_origins,103
4,snapping_diagnostic_gt_100m_count,87



outputs/tables/origin_cleaning_summary.csv


,origin_set,n_origins,removed_due_to_snap_gt_100m,removed_due_to_snap_gt_250m,removed_due_to_low_overlap,removed_total,share_removed,median_snap_distance_m,max_snap_distance_m,n_unassigned_districts
0,full,475,0,0,0,0,0.000000,26.919742,881.906311,0
1,clean100,387,87,29,36,88,0.185263,20.232850,99.804004,0
2,clean250,428,46,29,36,47,0.098947,23.819101,236.005790,0



outputs/tables/robustness_summary_by_origin_set.csv


,origin_set,amenity_type,n_origins,official_share_accessible_15,osm_share_accessible_15,difference_share_percentage_points,total_disagreement_share,osm_false_access_share,osm_hidden_access_share,median_difference_minutes,mean_difference_minutes
0,full,library,475,0.414737,0.440000,2.526316,0.033684,0.029474,0.004211,0.000000,-0.806823
1,full,playground,475,0.646316,0.808421,16.210526,0.162105,0.162105,0.000000,-3.319791,-6.284170
2,full,sports_facility,475,0.492632,0.667368,17.473684,0.275789,0.225263,0.050526,-0.889169,-3.493436
3,clean100,library,387,0.498708,0.527132,2.842377,0.038760,0.033592,0.005168,0.000000,-0.842182
4,clean100,playground,387,0.731266,0.899225,16.795866,0.167959,0.167959,0.000000,-3.097500,-5.862887
5,clean100,sports_facility,387,0.565891,0.754522,18.863049,0.302326,0.245478,0.056848,-0.935064,-3.444450
6,clean250,library,428,0.455607,0.483645,2.803738,0.037383,0.032710,0.004673,0.000000,-0.809515
7,clean250,playground,428,0.691589,0.862150,17.056075,0.170561,0.170561,0.000000,-3.223597,-6.219813
8,clean250,sports_facility,428,0.542056,0.724299,18.224299,0.294393,0.238318,0.056075,-0.856650,-3.374774
